# Synthetic data for testing a single-hidden-layer MLP

This code was generated by Claude.

We generate regression data from a **known** ReLU network so we can test a from-scratch NumPy backprop implementation against a problem whose best achievable test error is known in advance.

**Decisions**

| Item | Choice |
|---|---|
| Inputs | $X \sim N(0, I)$, 4 features |
| Hidden layer | 6 ReLU units |
| Output | 1 linear unit (no activation) → regression |
| Biases | augmented form: prepend a column of 1s before each layer; row 0 of each weight matrix holds the biases |
| Weight shapes | $\tilde W_1$: 5 × 6, $\tilde W_2$: 7 × 1 |
| Weight values | every entry = random sign × Uniform(0.5, 1.5) |
| Noise | $\varepsilon \sim N(0, 0.1^2)$ |
| Sizes | 10,000 train, 10,000 test |
| Seed | fixed (42) |

**The true network**

$$
\begin{aligned}
\tilde X &= [\mathbf{1} \mid X] && (n \times 5) \\
H &= \text{relu}(\tilde X\, \tilde W_1) && (n \times 6) \\
\tilde H &= [\mathbf{1} \mid H] && (n \times 7) \\
f &= \tilde H\, \tilde W_2 && (n \times 1) \quad \text{noise-free target} \\
y &= f + \varepsilon && (n \times 1) \quad \text{observed target}
\end{aligned}
$$

## 1. Imports

In [16]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=3, suppress=True)

## 2. Building blocks

### 2a. `relu()`
Elementwise $\max(0, z)$.

In [2]:
def relu(z):
    """Elementwise ReLU: max(0, z)."""
    return np.maximum(0.0, z)

# quick test
relu(np.array([-2.0, -0.5, 0.0, 0.5, 2.0]))

array([0. , 0. , 0. , 0.5, 2. ])

### 2b. `add_bias_column()`
Prepends a column of 1s, turning an $n \times k$ matrix into $n \times (k+1)$. Multiplying by a weight matrix whose row 0 holds the biases then adds the bias to every row.

In [3]:
def add_bias_column(A):
    """Prepend a column of 1s: (n x k) -> (n x (k+1))."""
    return np.hstack([np.ones((A.shape[0], 1)), A])

# quick test
A = np.array([[2.0, 3.0],
              [4.0, 5.0]])
add_bias_column(A)

array([[1., 2., 3.],
       [1., 4., 5.]])

**Check that the augmented form equals "weights plus a separate bias".** Below, `W_aug` stacks the bias `b` on top of `W`, and both ways give the same answer.

In [4]:
W = np.array([[1.0, -1.0, 0.5],
              [2.0,  0.0, 1.0]])   # 2 inputs -> 3 units
b = np.array([10.0, 20.0, 30.0])   # one bias per unit

W_aug = np.vstack([b, W])          # 3 x 3, row 0 = biases

print("A @ W + b          :\n", A @ W + b)
print("[1|A] @ W_aug      :\n", add_bias_column(A) @ W_aug)

A @ W + b          :
 [[18. 18. 34.]
 [24. 16. 37.]]
[1|A] @ W_aug      :
 [[18. 18. 34.]
 [24. 16. 37.]]


### 2c. `random_sign_uniform()`
Each entry is a random sign times Uniform(0.5, 1.5), so every weight is between 0.5 and 1.5 in size. None can be close to zero.

In [5]:
def random_sign_uniform(rng, shape, low=0.5, high=1.5):
    """Entries are (random sign) x Uniform(low, high)."""
    signs = rng.choice([-1.0, 1.0], size=shape)
    magnitudes = rng.uniform(low, high, size=shape)
    return signs * magnitudes

# quick test
random_sign_uniform(np.random.default_rng(0), (2, 5))

array([[ 1.413,  1.107,  1.229, -1.044, -1.435],
       [-1.316, -0.503, -1.357, -0.534,  1.23 ]])

### 2d. `forward()`
The forward pass of the network in augmented form. It returns the output `f` and the hidden activations `H`.

In [6]:
def forward(X, W1, W2):
    """
    Forward pass of a 1-hidden-layer ReLU net in augmented form.
    Returns (f, H): f is (n x 1), H is (n x n_hidden).
    """
    Z1 = add_bias_column(X) @ W1      # (n x n_hidden) pre-activations
    H = relu(Z1)                      # (n x n_hidden)
    f = add_bias_column(H) @ W2       # (n x 1)
    return f, H

# quick shape test with 3 rows of random input
_rng = np.random.default_rng(0)
_X = _rng.standard_normal((3, 4)) # feature observations
_W1 = random_sign_uniform(_rng, (5, 6)) # parameter matrix hidden layer
_W2 = random_sign_uniform(_rng, (7, 1)) # parameter matrix output layer
_f, _H = forward(_X, _W1, _W2)
print("f shape:", _f.shape, "  H shape:", _H.shape)
print("H (note the zeros from ReLU):\n", _H)

f shape: (3, 1)   H shape: (3, 6)
H (note the zeros from ReLU):
 [[0.    0.877 0.496 0.    1.61  1.857]
 [0.    1.703 0.352 0.    2.452 2.123]
 [0.    3.773 3.484 0.    2.07  0.   ]]


### 2e. `mse()` and `fit_linear_regression()`

In [7]:
def mse(y_true, y_pred):
    """Mean squared error; works for (n,) or (n x 1) arrays."""
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return float(np.mean((y_true - y_pred) ** 2))


def fit_linear_regression(X, y):
    """Least-squares fit of y on [1 | X]. Returns beta of shape ((k+1) x 1)."""
    beta, *_ = np.linalg.lstsq(add_bias_column(X), y, rcond=None)
    return beta

# quick test: recover y = 1 + 2x exactly
_x = np.arange(5.0).reshape(-1, 1)
fit_linear_regression(_x, 1 + 2 * _x)

array([[1.],
       [2.]])

In [8]:
_x

array([[0.],
       [1.],
       [2.],
       [3.],
       [4.]])

## 3. Settings

Change these to experiment. For example, raise `SIGMA` for a harder problem or change `SEED` for a different true network.

In [9]:
SEED      = 42
N_TRAIN   = 10_000
N_TEST    = 10_000
N_INPUTS  = 4
N_HIDDEN  = 6
SIGMA     = 0.1

WEIGHT_LOW, WEIGHT_HIGH = 0.5, 1.5
ACTIVE_RANGE     = (0.10, 0.90)  # each hidden unit must be "on" for 10%-90% of rows
MIN_LINEAR_RATIO = 5.0           # linear-regression test MSE must be >= 5 * sigma^2
MAX_DRAWS        = 1_000

## 4. Draw inputs and noise

These don't depend on the weights, so we draw them once. All randomness comes from one seeded generator, so re-running from this cell gives the same data.

In [10]:
rng = np.random.default_rng(SEED)

X_train = rng.standard_normal((N_TRAIN, N_INPUTS))
X_test  = rng.standard_normal((N_TEST,  N_INPUTS))
eps_train = SIGMA * rng.standard_normal((N_TRAIN, 1))
eps_test  = SIGMA * rng.standard_normal((N_TEST,  1))

print("X_train:", X_train.shape, "  X_test:", X_test.shape)
print("noise std (train):", eps_train.std().round(4))

X_train: (10000, 4)   X_test: (10000, 4)
noise std (train): 0.1009


## 5. Draw the true weights, redrawing until both checks pass

1. **Every hidden unit switches on and off within the data.** It must be active on 10%–90% of training rows. A unit that is always on behaves linearly, and one that is always off is a constant.
2. **A linear model can't get close to the noise floor.** Its test MSE must be at least 5σ². This confirms the data really needs the hidden layer.

In [11]:
lo, hi = ACTIVE_RANGE

for draw in range(1, MAX_DRAWS + 1):
    W1 = random_sign_uniform(rng, (N_INPUTS + 1, N_HIDDEN), WEIGHT_LOW, WEIGHT_HIGH)
    W2 = random_sign_uniform(rng, (N_HIDDEN + 1, 1),        WEIGHT_LOW, WEIGHT_HIGH)

    # Check 1: each hidden unit is active on 10%-90% of training rows
    active_frac = (add_bias_column(X_train) @ W1 > 0).mean(axis=0)
    if np.any(active_frac < lo) or np.any(active_frac > hi):
        print(f"draw {draw}: failed check 1 (active fractions {active_frac})")
        continue

    # Build targets for this draw
    f_train, _ = forward(X_train, W1, W2)
    f_test,  _ = forward(X_test,  W1, W2)
    y_train = f_train + eps_train
    y_test  = f_test  + eps_test

    # Check 2: linear regression can't reach the noise floor
    beta = fit_linear_regression(X_train, y_train)
    linear_test_mse = mse(y_test, add_bias_column(X_test) @ beta)
    if linear_test_mse < MIN_LINEAR_RATIO * SIGMA**2:
        print(f"draw {draw}: failed check 2 (linear test MSE {linear_test_mse:.4f})")
        continue

    print(f"draw {draw}: passed both checks")
    break
else:
    raise RuntimeError("No draw passed; loosen ACTIVE_RANGE or MIN_LINEAR_RATIO.")

draw 1: passed both checks


## 6. Inspect the true network and the data

In [12]:
print("W1 (row 0 = hidden biases, rows 1-4 = input weights):")
print(W1)
print("\nW2 transposed (entry 0 = output bias, entries 1-6 = hidden-to-output weights):")
print(W2.T)
print("\nFraction of training rows where each hidden unit is active:")
print(active_frac)

W1 (row 0 = hidden biases, rows 1-4 = input weights):
[[-0.894 -0.922 -0.888  1.259 -0.797  1.   ]
 [-0.774  1.316 -1.475  0.589 -0.949  1.138]
 [-0.832  0.726  0.574  0.985  0.829 -0.525]
 [-1.044 -1.146 -1.457  0.956 -0.862 -0.662]
 [-1.233 -0.506  1.428  0.741 -0.625  1.062]]

W2 transposed (entry 0 = output bias, entries 1-6 = hidden-to-output weights):
[[-0.567  1.486  0.977  0.739  1.102 -0.897  0.593]]

Fraction of training rows where each hidden unit is active:
[0.327 0.315 0.37  0.775 0.325 0.713]


In [27]:
# saving out the true weights matrix for experimenting
np.save("W1", W1)
np.save("W2", W2)

In [13]:
print("Shapes:")
print("  X_train", X_train.shape, "  y_train", y_train.shape, "  f_train", f_train.shape)
print("  X_test ", X_test.shape,  "  y_test ", y_test.shape,  "  f_test ", f_test.shape)
print(f"\nVar(y_train) = {y_train.var():.3f}")
print(f"Signal-to-noise  Var(f)/sigma^2 = {f_train.var() / SIGMA**2:.0f}")
print("\nFirst 5 rows of y_train:", y_train[:5].ravel())

Shapes:
  X_train (10000, 4)   y_train (10000, 1)   f_train (10000, 1)
  X_test  (10000, 4)   y_test  (10000, 1)   f_test  (10000, 1)

Var(y_train) = 4.072
Signal-to-noise  Var(f)/sigma^2 = 406

First 5 rows of y_train: [2.934 2.795 2.48  1.859 1.629]


In [20]:
df_train = pd.DataFrame(
    np.hstack([X_train, y_train]),
    columns=["feature_1", "feature_2", "feature_3", "feature_4", "label"]
)
df_train

,feature_1,feature_2,feature_3,feature_4,label
0,0.304717,-1.039984,0.750451,0.940565,2.934120
1,-1.951035,-1.302180,0.127840,-0.316243,2.795016
2,-0.016801,-0.853044,0.879398,0.777792,2.480063
3,0.066031,1.127241,0.467509,-0.859292,1.859139
4,0.368751,-0.958883,0.878450,-0.049926,1.628915
...,...,...,...,...,...
9995,-0.988813,-0.349446,-0.060661,-0.512423,0.483559
9996,0.610067,0.935540,-1.732124,-0.145460,4.202910
9997,-2.903122,-0.091574,-0.102752,1.776210,3.824204
9998,1.476257,0.932350,-0.959739,-1.017508,5.135577


In [21]:
df_test = pd.DataFrame(
    np.hstack([X_test, y_test]),
    columns=["feature_1", "feature_2", "feature_3", "feature_4", "label"]
)
df_test

,feature_1,feature_2,feature_3,feature_4,label
0,-0.899768,-1.258172,0.332616,0.153343,0.086122
1,-2.224559,-1.301935,0.255188,-0.072214,2.934076
2,-0.400932,0.605678,-1.443033,-0.594992,1.628798
3,0.518037,-0.649295,0.312547,-1.146230,0.800906
4,-0.823849,-1.782198,-2.057127,-0.475947,7.073593
...,...,...,...,...,...
9995,0.272676,-1.490224,0.342113,-0.080671,0.829739
9996,-0.227261,-1.302346,0.875833,0.921119,2.089276
9997,-1.711401,-0.599959,-0.286847,0.339636,2.083228
9998,0.143236,-0.071576,0.485829,1.866900,5.158848


In [22]:
df_train.to_csv("nn_train.csv", index=False)
df_test.to_csv("nn_test.csv", index=False)

## 7. Baselines and the noise floor

The **noise floor** σ² = 0.01 is the best test MSE any model can expect, because the noise can't be predicted. A working MLP should land just above it. The two baselines show where a model with no hidden layer ends up.

In [23]:
noise_floor = SIGMA**2
mean_baseline_mse = mse(y_test, np.full_like(y_test, y_train.mean()))

print(f"noise floor (sigma^2)        : {noise_floor:.5f}")
print(f"baseline: predict the mean   : {mean_baseline_mse:.5f}")
print(f"baseline: linear regression  : {linear_test_mse:.5f}")

noise floor (sigma^2)        : 0.01000
baseline: predict the mean   : 4.03576
baseline: linear regression  : 2.11510


## 8. `evaluate()`: score a model's predictions

Pass in predictions on `X_test` (and optionally on `X_train`). Besides the usual test MSE, it reports **MSE against the noise-free `f_test`**. That comparison is only possible with simulated data: a good fit should bring it close to 0.

Don't compare fitted weights to `W1` and `W2` directly. Hidden units can be reordered and positively rescaled without changing the function.

In [24]:
def evaluate(y_hat_test, y_hat_train=None):
    results = {
        "noise floor (sigma^2)":            noise_floor,
        "baseline: predict mean":           mean_baseline_mse,
        "baseline: linear regression":      linear_test_mse,
        "your model: test MSE":             mse(y_test, y_hat_test),
        "your model: MSE vs true f (test)": mse(f_test, y_hat_test),
    }
    if y_hat_train is not None:
        results["your model: train MSE"] = mse(y_train, y_hat_train)

    width = max(len(k) for k in results)
    print("-" * (width + 14))
    for k, v in results.items():
        print(f"{k:<{width}}  {v:>10.5f}")
    print("-" * (width + 14))

    ratio = results["your model: test MSE"] / noise_floor
    if ratio < 0.9:
        print("Test MSE is well BELOW the noise floor -> check for leakage / a bug.")
    elif ratio <= 1.1:
        print(f"Test MSE is within 10% of the noise floor ({ratio:.3f}x) -> fit looks good.")
    else:
        print(f"Test MSE is {ratio:.2f}x the noise floor -> not fully fit yet "
              "(more training, more hidden units, a new init, or a bug).")
    return results

## 9. Sanity check: score the true network

Treat the true weights as if they were a fitted model. Test MSE should be about 0.01, and MSE against `f_test` should be exactly 0. This is the best your backprop can do.

In [25]:
f_hat_test,  _ = forward(X_test,  W1, W2)
f_hat_train, _ = forward(X_train, W1, W2)
_ = evaluate(f_hat_test, f_hat_train)

----------------------------------------------
noise floor (sigma^2)                0.01000
baseline: predict mean               4.03576
baseline: linear regression          2.11510
your model: test MSE                 0.01010
your model: MSE vs true f (test)     0.00000
your model: train MSE                0.01019
----------------------------------------------
Test MSE is within 10% of the noise floor (1.010x) -> fit looks good.


For comparison, score the **linear regression baseline** the same way. It is far above the noise floor.

In [26]:
_ = evaluate(add_bias_column(X_test) @ beta, add_bias_column(X_train) @ beta)

----------------------------------------------
noise floor (sigma^2)                0.01000
baseline: predict mean               4.03576
baseline: linear regression          2.11510
your model: test MSE                 2.11510
your model: MSE vs true f (test)     2.10326
your model: train MSE                2.03139
----------------------------------------------
Test MSE is 211.51x the noise floor -> not fully fit yet (more training, more hidden units, a new init, or a bug).


## 10. Your model goes here

Suggested order:
1. Fit with about **16 hidden units** first. It's easier to optimize and has the same noise floor, so it tests whether your code is correct.
2. Then fit with exactly **6**, from several random starts. Some runs may get stuck above 0.01.

Then call `evaluate(y_hat_test, y_hat_train)`.

In [ ]:
# Example skeleton -- replace with your own training code.
#
# W1_hat, W2_hat = train_my_mlp(X_train, y_train, n_hidden=16, ...)
# y_hat_train, _ = forward(X_train, W1_hat, W2_hat)
# y_hat_test,  _ = forward(X_test,  W1_hat, W2_hat)
# evaluate(y_hat_test, y_hat_train)